# Лабораторная работа №4

In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import (BaggingClassifier, BaggingRegressor,
                              RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor,
                              AdaBoostClassifier, AdaBoostRegressor,
                              StackingClassifier, StackingRegressor)
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_squared_error, mean_absolute_error, r2_score, confusion_matrix, RocCurveDisplay, classification_report)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from py_boost import GradientBoosting

import optuna

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [17]:
import sys
print(sys.executable)

/usr/bin/python3


### Загрузка датасетов

In [22]:
from google.colab import files
uploaded_wine = files.upload()

df_wine = pd.read_csv('final_data_wine.csv', index_col=0)
df_wine

Saving final_data_wine.csv to final_data_wine (2).csv


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,0.164343,2.283475,-2.279442,-0.749952,1.238001,-1.133138,-1.467934,1.092436,1.808131,0.263930,-0.938658,5,1
1,0.477600,3.405904,-2.279442,-0.601255,2.344580,-0.322206,-0.876417,0.751612,-0.125112,1.160428,-0.603680,5,1
2,0.477600,2.657618,-1.988649,-0.664982,2.042786,-0.901443,-1.109439,0.819769,0.249064,0.936303,-0.603680,5,1
3,3.140279,-0.335528,1.791656,-0.749952,1.187702,-0.785596,-1.001890,1.160613,-0.374562,0.413346,-0.603680,6,1
4,0.164343,2.283475,-2.279442,-0.749952,1.238001,-1.133138,-1.467934,1.092436,1.808131,0.263930,-0.938658,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4893,-0.775426,-0.772028,-0.171195,-0.813679,-0.623063,-0.380130,-0.428298,-1.177475,0.311427,-0.184319,0.568742,6,0
4894,-0.462170,-0.086099,0.337693,0.545837,-0.220671,1.531351,0.933984,0.104033,-0.436925,-0.483152,-0.771169,5,0
4895,-0.540484,-0.584956,-0.898176,-0.898649,-0.522465,-0.032588,-0.087728,-0.700317,-1.434728,-0.483152,-0.938658,6,0
4896,-1.323625,-0.273170,-0.098496,-0.919892,-1.478147,-0.611825,-0.105652,-2.012500,0.747965,-1.080818,1.908654,7,0


In [ ]:
df_wine.info()

In [23]:
from google.colab import files
uploaded_wine = files.upload()

df_diamonds = pd.read_csv('final_data_diamonds.csv', index_col=0)
df_diamonds

Saving diamonds.csv to diamonds.csv


,carat,cut,color,clarity,depth,table,price,x,y,z
0,-1.199341,4,5,1,-0.176252,-1.096824,326,-1.587838,-1.573374,-1.593630
1,-1.241582,3,5,2,-1.381332,1.591635,326,-1.641325,-1.699003,-1.766179
2,-1.199341,1,5,4,-3.437054,3.383941,327,-1.498691,-1.492612,-1.766179
3,-1.072620,3,1,3,0.461732,0.247406,334,-1.364972,-1.349036,-1.306050
4,-1.030379,1,0,1,1.099714,0.247406,335,-1.240166,-1.241354,-1.133502
...,...,...,...,...,...,...,...,...,...,...
53935,-0.164447,4,6,2,-0.672462,-0.200671,2757,0.016798,0.023908,-0.055076
53936,-0.164447,1,6,2,0.957939,-1.096824,2757,-0.036690,0.014935,0.103093
53937,-0.206688,2,6,2,0.745278,1.143559,2757,-0.063434,-0.047880,0.031198
53938,0.131237,3,2,1,-0.530687,0.247406,2757,0.373383,0.346954,0.290021


In [24]:
df_diamonds.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  int64  
 2   color    53940 non-null  int64  
 3   clarity  53940 non-null  int64  
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y        53940 non-null  float64
 9   z        53940 non-null  float64
dtypes: float64(6), int64(4)
memory usage: 4.5 MB


### Выделеение целевого признака и предиктора, разделение на выборки

In [25]:
X_wine = df_wine.drop(columns=['type'])
y_wine = df_wine['type']
print(X_wine.shape, y_wine.shape)
print(y_wine.value_counts())

(5974, 12) (5974,)
type
0    4575
1    1399
Name: count, dtype: int64


In [26]:
X_diamonds = df_diamonds.drop(columns=["price"])
y_diamonds = df_diamonds['price']
print(X_diamonds.shape, y_diamonds.shape)
print(y_diamonds.value_counts())

(53940, 9) (53940,)
price
605      132
802      127
625      126
828      125
776      124
        ... 
13061      1
13074      1
13075      1
13077      1
13078      1
Name: count, Length: 11602, dtype: int64


### Разделение данных

In [27]:
X_temp_wine, X_test_wine, y_temp_wine, y_test_wine = train_test_split(X_wine, y_wine, test_size=0.2, stratify=y_wine, random_state=RNG_SEED)
X_train_wine, X_val_wine, y_train_wine, y_val_wine = train_test_split(X_temp_wine, y_temp_wine, test_size=0.25, stratify=y_temp_wine, random_state=RNG_SEED)

print(X_train_wine.shape, X_val_wine.shape, X_test_wine.shape)
print(y_train_wine.shape, X_val_wine.shape, y_test_wine.shape)


(3584, 12) (1195, 12) (1195, 12)
(3584,) (1195, 12) (1195,)


In [28]:
X_temp_diamonds, X_test_diamonds, y_temp_diamonds, y_test_diamonds = train_test_split(X_diamonds, y_diamonds, test_size=0.2, random_state=RNG_SEED)
X_train_diamonds, X_val_diamonds, y_train_diamonds, y_val_diamonds = train_test_split(X_temp_diamonds, y_temp_diamonds, test_size=0.25, random_state=RNG_SEED)

print(X_train_diamonds.shape, X_val_diamonds.shape, X_test_diamonds.shape)
print(y_train_diamonds.shape, X_val_diamonds.shape, y_test_diamonds.shape)

(32364, 9) (10788, 9) (10788, 9)
(32364,) (10788, 9) (10788,)


### Балансировка

In [29]:
y_train_wine.value_counts()

,count
type,
0,2745
1,839


In [30]:
oversample = SMOTE()

X_train_bal_wine, y_train_bal_wine = oversample.fit_resample(X_train_wine, y_train_wine)
y_train_bal_wine.value_counts()

,count
type,
0,2745
1,2745


### Вспомогательные функции

In [31]:
def calculate_metrics_cls(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted'),
        'Recall': recall_score(y_true, y_pred, average='weighted'),
        'F1': f1_score(y_true, y_pred, average='weighted')
    }

def calculate_metrics_reg(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MSE': mean_squared_error(y_true, y_pred),
    }

def fmt(val, metric_name):
    if 'R2' in metric_name or 'F1' in metric_name:
        prec = 2
    else:
        prec = 4
    return f"{val:.{prec}f}"


def plot_and_extract_rules(model, X_train, feature_names):
    plt.figure(figsize=(20, 10))
    plot_tree(model, feature_names=feature_names, class_names=['White', 'Red'],
              filled=True, max_depth=3, fontsize=10)
    plt.title("Decision Tree Visualization")
    plt.show()

    from sklearn.tree import _tree

    tree = model.tree_
    rules = []

    def extract_rules(node, conditions):
        if tree.feature[node] != _tree.TREE_UNDEFINED:
            feature = feature_names[tree.feature[node]]
            threshold = tree.threshold[node]

            extract_rules(tree.children_left[node],
                         conditions + [f"{feature} <= {threshold:.2f}"])
            extract_rules(tree.children_right[node],
                         conditions + [f"{feature} > {threshold:.2f}"])
        else:
            class_name = 'Red' if tree.value[node][0][1] > tree.value[node][0][0] else 'White'
            rules.append(f"IF {' AND '.join(conditions)} THEN {class_name}")

    extract_rules(0, [])

    for rule in rules:
        print(f"  {rule}")

In [54]:
cls_models = {
    'SketchBoost': GradientBoosting(
        loss='logloss',
        ntrees=100,
        lr=0.1,
        max_depth=4,
        seed=RNG_SEED,
        verbose=1
    )
}

reg_models = {
    'SketchBoost': GradientBoosting(
        loss='mse',
        ntrees=100,
        lr=0.1,
        max_depth=4,
        seed=RNG_SEED,
        verbose=0
    )
}

cv_wine = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)
cv_diamonds = KFold(n_splits=5, shuffle=True, random_state=RNG_SEED)

cls_results_wine = []
reg_results_diamonds = []

In [56]:
sb_cls = GradientBoosting(loss='logloss', ntrees=100, lr=0.1, max_depth=4, seed=RNG_SEED)
sb_cls.fit(X_train_bal_wine, y_train_bal_wine)
y_pred_train = (sb_cls.predict(X_train_bal_wine) >= 0.5).astype(int)
y_pred_test = (sb_cls.predict(X_test_wine) >= 0.5).astype(int)

metrics_train = calculate_metrics_cls(y_train_bal_wine, y_pred_train)
metrics_test = calculate_metrics_cls(y_test_wine, y_pred_test)

row = {'Algorithm': 'SketchBoost'}
for k in metrics_train.keys():
    row[('Train Data', k)] = fmt(metrics_train[k], k)
    row[('Test Data', k)] = fmt(metrics_test[k], k)
for k in ['Accuracy', 'Precision', 'Recall', 'F1']:
    row[('K-Fold CV', f'{k} (mean±std)')] = 'N/A'

cls_results_wine.append(row)

[18:25:33] Stdout logging level is INFO.


INFO:py_boost.callbacks.callback:Stdout logging level is INFO.


[18:25:33] GDBT train starts. Max iter 100, early stopping rounds 100


INFO:py_boost.callbacks.callback:GDBT train starts. Max iter 100, early stopping rounds 100


[18:25:33] Iter 0; 


...

INFO:py_boost.callbacks.callback:Iter 80; 


[18:25:35] Iter 90; 


INFO:py_boost.callbacks.callback:Iter 90; 


[18:25:35] Iter 99; 


INFO:py_boost.callbacks.callback:Iter 99; 


In [57]:
sb_reg = GradientBoosting(loss='mse', ntrees=100, lr=0.1, max_depth=4, seed=RNG_SEED)
sb_reg.fit(X_train_diamonds, y_train_diamonds)
y_pred_train = sb_reg.predict(X_train_diamonds)
y_pred_test = sb_reg.predict(X_test_diamonds)

metrics_train = calculate_metrics_reg(y_train_diamonds, y_pred_train)
metrics_test = calculate_metrics_reg(y_test_diamonds, y_pred_test)

row = {'Algorithm': 'SketchBoost'}
for k in metrics_train.keys():
    row[('Train Data', k)] = fmt(metrics_train[k], k)
    row[('Test Data', k)] = fmt(metrics_test[k], k)
for k in ['R2', 'RMSE', 'MAE', 'MSE']:
    row[('K-Fold CV', f'{k} (mean±std)')] = 'N/A'

reg_results_diamonds.append(row)

[18:25:38] Stdout logging level is INFO.


INFO:py_boost.callbacks.callback:Stdout logging level is INFO.


[18:25:38] GDBT train starts. Max iter 100, early stopping rounds 100


INFO:py_boost.callbacks.callback:GDBT train starts. Max iter 100, early stopping rounds 100


[18:25:39] Iter 0; 


...

INFO:py_boost.callbacks.callback:Iter 80; 


[18:25:40] Iter 90; 


INFO:py_boost.callbacks.callback:Iter 90; 


[18:25:40] Iter 99; 


INFO:py_boost.callbacks.callback:Iter 99; 


In [65]:
def build_and_display_tables(results_list, task_name, metrics_order):
    print(f"\n{task_name}")

    df = pd.DataFrame(results_list).set_index('Algorithm')
    df.columns = pd.MultiIndex.from_tuples(df.columns)

    train_cols = [c for c in df.columns if c[0] == 'Train Data']
    train_cols_sorted = sorted(train_cols, key=lambda c: metrics_order.index(c[1]))

    test_cols = [c for c in df.columns if c[0] == 'Test Data']
    test_cols_sorted = sorted(test_cols, key=lambda c: metrics_order.index(c[1]))

    flat_data = []
    for algo in df.index:
        row = {'Algorithm': algo}
        for col in train_cols_sorted:
            row[f'Train {col[1]}'] = df.loc[algo, col]
        for col in test_cols_sorted:
            row[f'Test {col[1]}'] = df.loc[algo, col]
        flat_data.append(row)

    flat_df = pd.DataFrame(flat_data).set_index('Algorithm')
    display(flat_df)

    return flat_df

In [66]:
cls_metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
reg_metrics = ['R2', 'RMSE', 'MAE', 'MSE']

tbl_cls = build_and_display_tables(cls_results_wine, "Классификация (Wine)", cls_metrics)



Классификация (Wine)


,Train Accuracy,Train Precision,Train Recall,Train F1,Test Accuracy,Test Precision,Test Recall,Test F1
Algorithm,,,,,,,,
SketchBoost,1.0000,1.0000,1.0000,1.00,0.9933,0.9933,0.9933,0.99



Регрессия (Diamonds)


,Train R2,Train RMSE,Train MAE,Train MSE,Test R2,Test RMSE,Test MAE,Test MSE
Algorithm,,,,,,,,
SketchBoost,0.98,531.0348,288.3825,281997.9062,0.98,554.8798,294.9687,307891.6250
